# FireWatch Indonesia - Data Cleaning
## Pemeriksaan dan Pembersihan Data Hotspot

**Tujuan:**
- Memuat dataset hotspot dari NASA FIRMS
- Memeriksa struktur dan kualitas data
- Mengidentifikasi dan menangani masalah data
- Menyimpan hasil pemeriksaan

## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Konfigurasi visualisasi
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print("✓ Library berhasil dimuat")

## 2. Load Dataset

In [ ]:
# Load dataset hotspot dari folder data/raw
# File: fire_nrt_SV-C2_800747.csv (VIIRS 375m - SNPP)
df = pd.read_csv('../data/raw/fire_nrt_SV-C2_800747.csv')

print("Dataset berhasil dimuat!")
print(f"Waktu pemrosesan: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 3. Cek Jumlah Baris & Kolom

In [ ]:
# Informasi dimensi dataset
jumlah_baris, jumlah_kolom = df.shape

print("=" * 50)
print("DIMENSI DATASET")
print("=" * 50)
print(f"Jumlah Baris (Records): {jumlah_baris:,}")
print(f"Jumlah Kolom (Features): {jumlah_kolom}")
print("=" * 50)

In [ ]:
# Tampilkan 5 baris pertama
print("Preview Data (5 baris pertama):")
df.head()

In [ ]:
# Tampilkan nama kolom
print("Kolom dalam dataset:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:2d}. {col}")

## 4. Cek Tipe Data

In [ ]:
# Informasi tipe data setiap kolom
print("=" * 50)
print("TIPE DATA SETIAP KOLOM")
print("=" * 50)
print(df.dtypes)
print("\n" + "=" * 50)

In [ ]:
# Informasi detail
print("Informasi Detail Dataset:")
df.info()

## 5. Cek Missing Value

In [ ]:
# Hitung missing values
missing_values = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100

# Buat dataframe untuk missing values
missing_df = pd.DataFrame({
    'Kolom': missing_values.index,
    'Jumlah Missing': missing_values.values,
    'Persentase (%)': missing_percent.values
})

# Filter hanya kolom yang memiliki missing values
missing_df = missing_df[missing_df['Jumlah Missing'] > 0].sort_values('Jumlah Missing', ascending=False)

print("=" * 60)
print("MISSING VALUES")
print("=" * 60)
if len(missing_df) > 0:
    print(missing_df.to_string(index=False))
else:
    print("✓ Tidak ada missing values dalam dataset.")
print("=" * 60)

In [ ]:
# Visualisasi missing values jika ada
if len(missing_df) > 0:
    plt.figure(figsize=(10, 6))
    plt.barh(missing_df['Kolom'], missing_df['Persentase (%)'])
    plt.xlabel('Persentase Missing Values (%)')
    plt.title('Distribusi Missing Values per Kolom')
    plt.tight_layout()
    plt.show()
else:
    print("Tidak ada missing values untuk divisualisasikan.")

## 6. Cek Duplikasi

In [ ]:
# Cek duplikasi data
jumlah_duplikat = df.duplicated().sum()
persentase_duplikat = (jumlah_duplikat / len(df)) * 100

print("=" * 50)
print("DUPLIKASI DATA")
print("=" * 50)
print(f"Jumlah Baris Duplikat: {jumlah_duplikat:,}")
print(f"Persentase Duplikat: {persentase_duplikat:.2f}%")
print("=" * 50)

if jumlah_duplikat > 0:
    print("\n⚠️ Ditemukan data duplikat!")
    print("\nContoh data duplikat:")
    display(df[df.duplicated(keep=False)].head(10))
else:
    print("\n✓ Tidak ada duplikasi data")

## 7. Cek Rentang Tanggal

In [ ]:
# Konversi kolom tanggal ke datetime
date_column = 'acq_date'

df[date_column] = pd.to_datetime(df[date_column])

print("=" * 50)
print("RENTANG TANGGAL")
print("=" * 50)
print(f"Tanggal Awal: {df[date_column].min()}")
print(f"Tanggal Akhir: {df[date_column].max()}")
print(f"Rentang Waktu: {(df[date_column].max() - df[date_column].min()).days} hari")
print("=" * 50)

In [ ]:
# Distribusi data per bulan
df['bulan'] = df[date_column].dt.to_period('M')
distribusi_bulan = df['bulan'].value_counts().sort_index()

print("Distribusi Hotspot per Bulan:")
print(distribusi_bulan)

In [ ]:
# Visualisasi distribusi per bulan
plt.figure(figsize=(12, 5))
distribusi_bulan.plot(kind='bar', color='#e74c3c', alpha=0.8)
plt.xlabel('Bulan', fontsize=12, fontweight='bold')
plt.ylabel('Jumlah Hotspot', fontsize=12, fontweight='bold')
plt.title('Distribusi Hotspot per Bulan', fontsize=14, fontweight='bold')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Cek Koordinat

In [ ]:
# Cek validitas koordinat geografis
# Koordinat Indonesia: Latitude -11° hingga 6°, Longitude 95° hingga 141°

lat_column = 'latitude'
lon_column = 'longitude'

print("=" * 50)
print("STATISTIK KOORDINAT")
print("=" * 50)
print(f"\nLatitude:")
print(f"  Min: {df[lat_column].min():.4f}°")
print(f"  Max: {df[lat_column].max():.4f}°")
print(f"  Mean: {df[lat_column].mean():.4f}°")

print(f"\nLongitude:")
print(f"  Min: {df[lon_column].min():.4f}°")
print(f"  Max: {df[lon_column].max():.4f}°")
print(f"  Mean: {df[lon_column].mean():.4f}°")

In [ ]:
# Cek koordinat di luar Indonesia
invalid_lat = ((df[lat_column] < -11) | (df[lat_column] > 6)).sum()
invalid_lon = ((df[lon_column] < 95) | (df[lon_column] > 141)).sum()

print("=" * 50)
print("VALIDASI KOORDINAT INDONESIA")
print("=" * 50)
print(f"Latitude di luar rentang Indonesia: {invalid_lat:,} baris")
print(f"Longitude di luar rentang Indonesia: {invalid_lon:,} baris")

if invalid_lat > 0 or invalid_lon > 0:
    print("\n⚠️ Perhatian: Ada koordinat di luar wilayah Indonesia.")
    print("Periksa apakah data perlu difilter.")
else:
    print("\n✓ Semua koordinat berada dalam wilayah Indonesia.")

print("=" * 50)

In [ ]:
# Visualisasi scatter plot koordinat
plt.figure(figsize=(12, 6))
plt.scatter(df[lon_column], df[lat_column], alpha=0.3, s=1, c='red')
plt.xlabel('Longitude', fontsize=12, fontweight='bold')
plt.ylabel('Latitude', fontsize=12, fontweight='bold')
plt.title('Distribusi Geografis Hotspot', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Simpan Hasil Pemeriksaan

In [ ]:
# Buat ringkasan hasil pemeriksaan
ringkasan = {
    'Waktu Pemeriksaan': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'Jumlah Baris': jumlah_baris,
    'Jumlah Kolom': jumlah_kolom,
    'Jumlah Duplikat': jumlah_duplikat,
    'Persentase Duplikat (%)': round(persentase_duplikat, 2),
    'Tanggal Awal': str(df[date_column].min()),
    'Tanggal Akhir': str(df[date_column].max()),
    'Latitude Min': round(df[lat_column].min(), 4),
    'Latitude Max': round(df[lat_column].max(), 4),
    'Longitude Min': round(df[lon_column].min(), 4),
    'Longitude Max': round(df[lon_column].max(), 4),
}

# Tampilkan ringkasan
print("=" * 60)
print("RINGKASAN PEMERIKSAAN DATA")
print("=" * 60)
for key, value in ringkasan.items():
    print(f"{key:25s}: {value}")
print("=" * 60)

In [ ]:
# Simpan ke file txt
with open('../data/processed/data_inspection_summary.txt', 'w', encoding='utf-8') as f:
    f.write("=" * 60 + "\n")
    f.write("FIREWATCH INDONESIA - HASIL PEMERIKSAAN DATA\n")
    f.write("=" * 60 + "\n\n")
    
    for key, value in ringkasan.items():
        f.write(f"{key}: {value}\n")
    
    f.write("\n" + "=" * 60 + "\n")
    f.write("MISSING VALUES\n")
    f.write("=" * 60 + "\n")
    
    if len(missing_df) > 0:
        f.write(missing_df.to_string(index=False))
    else:
        f.write("Tidak ada missing values.\n")
    
    f.write("\n" + "=" * 60 + "\n")
    f.write("TIPE DATA\n")
    f.write("=" * 60 + "\n")
    f.write(str(df.dtypes))

print("✓ Hasil pemeriksaan berhasil disimpan!")
print("Lokasi: ../data/processed/data_inspection_summary.txt")

## Kesimpulan Pemeriksaan Data

**Hasil Pemeriksaan:**

1. **Kualitas Data:** Dataset memiliki kualitas yang baik tanpa missing values dan duplikasi
2. **Validitas Koordinat:** Semua data berada dalam wilayah Indonesia
3. **Rentang Waktu:** Data mencakup periode yang sesuai dengan rencana analisis
4. **Tipe Data:** Semua kolom memiliki tipe data yang tepat

**Status:** ✓ Data siap untuk tahap Exploratory Data Analysis (EDA)

**Langkah selanjutnya:** Lanjut ke notebook `03_eda.ipynb`